# Feature Engineering — Anomaly Detection

Combine price data + sentiment scores to build the feature matrix for XGBoost.

**Inputs:**
- `data/raw/prices/*.parquet` — 30 BIST30 tickers, 5-year daily OHLCV
- `data/processed/sentiment_scores.parquet` — 1,069 scored KAP disclosures

**Output:**
- `data/processed/features.parquet` — one row per ticker-date, all features + anomaly label

In [39]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta


## 1. Load Price Data

In [31]:
price_dir = Path("/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/data/raw/prices")
frames = []

for f in sorted(price_dir.glob("*.parquet")):
    ticker = f.stem
    tmp = pd.read_parquet(f)
    tmp.index.name = 'date'
    tmp = tmp.reset_index()
    tmp['ticker'] = ticker
    tmp['date'] = pd.to_datetime(tmp['date']).dt.tz_localize(None)
    frames.append(tmp)

prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"Price data: {len(prices):,} rows, {prices['ticker'].nunique()} tickers")
print(f"Columns: {prices.columns.tolist()}")
print(prices[prices['ticker'] == 'GARAN'].head())

Price data: 37,506 rows, 30 tickers
Columns: ['date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker']
Price       date     Close      High       Low      Open     Volume ticker
10002 2021-04-29  5.831884  5.872216  5.775421  5.783487  144520450  GARAN
10003 2021-04-30  5.815752  5.872216  5.799620  5.848017  116784823  GARAN
10004 2021-05-03  5.944812  5.944812  5.799620  5.815753  135618450  GARAN
10005 2021-05-04  5.864150  5.977077  5.856084  5.977077  140612336  GARAN
10006 2021-05-05  5.848018  5.896415  5.807686  5.880283  139863758  GARAN


In [32]:
print('ticker' in prices.columns)
print(prices.columns.tolist())

True
['date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker']


## 2. Price-Based Features

Per ticker, rolling window features computed on daily data.

In [33]:
from pathlib import Path
import pandas as pd
import numpy as np

price_dir = Path("/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/data/raw/prices")

frames = []
for f in sorted(price_dir.glob("*.parquet")):
    ticker = f.stem
    tmp = pd.read_parquet(f)
    tmp.index.name = 'date'
    tmp = tmp.reset_index()
    tmp['ticker'] = ticker
    tmp['date'] = pd.to_datetime(tmp['date']).dt.tz_localize(None)
    
    # Compute features per ticker right here
    tmp = tmp.sort_values('date')
    tmp['return_1d'] = tmp['Close'].pct_change()
    tmp['return_5d'] = tmp['Close'].pct_change(5)
    tmp['return_10d'] = tmp['Close'].pct_change(10)
    tmp['return_20d'] = tmp['Close'].pct_change(20)
    tmp['volatility_10d'] = tmp['return_1d'].rolling(10).std()
    tmp['volatility_20d'] = tmp['return_1d'].rolling(20).std()
    tmp['volume_ratio_20d'] = tmp['Volume'] / tmp['Volume'].rolling(20).mean()
    tmp['intraday_range'] = (tmp['High'] - tmp['Low']) / tmp['Close']
    tmp['gap'] = tmp['Open'] / tmp['Close'].shift(1) - 1
    
    frames.append(tmp)

prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

SECTOR_MAP = {
    "AKBNK": "Banking", "GARAN": "Banking", "HALKB": "Banking",
    "ISCTR": "Banking", "VAKBN": "Banking", "YKBNK": "Banking",
    "KCHOL": "Holding", "SAHOL": "Holding",
    "PGSUS": "Transportation", "TAVHL": "Transportation", "THYAO": "Transportation",
    "PETKM": "Chemicals", "SASA": "Chemicals",
    "TUPRS": "Chemicals", "GUBRF": "Chemicals",
    "ARCLK": "Consumer Durables", "FROTO": "Consumer Durables", "TOASO": "Consumer Durables",
    "BIMAS": "Retail", "MGROS": "Retail",
    "ASELS": "Defense", "CIMSA": "Construction",
    "EKGYO": "Real Estate", "TCELL": "Telecom",
    "EREGL": "Industrials", "KRDMD": "Industrials", "SISE": "Industrials",
    "KONTR": "Energy", "ODAS": "Energy",
    "TKFEN": "Industrials",
}
prices['sector'] = prices['ticker'].map(SECTOR_MAP)

print(f"Shape: {prices.shape}")
print(f"Columns: {prices.columns.tolist()}")
print(f"Tickers: {prices['ticker'].nunique()}")
print(prices[prices['ticker'] == 'GARAN'].head())

Shape: (37506, 17)
Columns: ['date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'volatility_10d', 'volatility_20d', 'volume_ratio_20d', 'intraday_range', 'gap', 'sector']
Tickers: 30
Price       date     Close      High       Low      Open     Volume ticker  \
10002 2021-04-29  5.831884  5.872216  5.775421  5.783487  144520450  GARAN   
10003 2021-04-30  5.815752  5.872216  5.799620  5.848017  116784823  GARAN   
10004 2021-05-03  5.944812  5.944812  5.799620  5.815753  135618450  GARAN   
10005 2021-05-04  5.864150  5.977077  5.856084  5.977077  140612336  GARAN   
10006 2021-05-05  5.848018  5.896415  5.807686  5.880283  139863758  GARAN   

Price  return_1d  return_5d  return_10d  return_20d  volatility_10d  \
10002        NaN        NaN         NaN         NaN             NaN   
10003  -0.002766        NaN         NaN         NaN             NaN   
10004   0.022192        NaN         NaN         NaN             NaN   
1

In [34]:
# Sector average daily return
sector_daily = prices.groupby(['date', 'sector'])['return_1d'].mean().reset_index()
sector_daily.columns = ['date', 'sector', 'sector_return_1d']

prices = prices.merge(sector_daily, on=['date', 'sector'], how='left')

# Relative return vs sector
prices['return_vs_sector'] = prices['return_1d'] - prices['sector_return_1d']

# Market-wide average return
market_daily = prices.groupby('date')['return_1d'].mean().reset_index()
market_daily.columns = ['date', 'market_return_1d']

prices = prices.merge(market_daily, on='date', how='left')
prices['return_vs_market'] = prices['return_1d'] - prices['market_return_1d']

print("Cross-sectional features added.")
print(f"Columns: {prices.columns.tolist()}")

Cross-sectional features added.
Columns: ['date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'volatility_10d', 'volatility_20d', 'volume_ratio_20d', 'intraday_range', 'gap', 'sector', 'sector_return_1d', 'return_vs_sector', 'market_return_1d', 'return_vs_market']


## 4. Sentiment Features

Merge sentiment scores with T+1 lag for after-hours announcements.

In [35]:
from datetime import timedelta

sent = pd.read_parquet("/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/data/processed/sentiment_scores.parquet")

# T+1 lag: after-hours announcements → next trading day
def get_trade_date(tarih):
    if tarih.hour >= 18 or tarih.hour < 10:
        next_day = tarih.normalize() + timedelta(days=1)
    else:
        next_day = tarih.normalize()
    while next_day.weekday() >= 5:
        next_day += timedelta(days=1)
    return next_day

sent['trade_date'] = sent['tarih'].apply(get_trade_date)

# Aggregate per ticker-date
sent_agg = sent.groupby(['ticker', 'trade_date']).agg(
    sent_mean=('sentiment', 'mean'),
    sent_max=('sentiment', 'max'),
    sent_min=('sentiment', 'min'),
    sent_count=('sentiment', 'count'),
    sent_confidence=('confidence', 'mean'),
).reset_index()
sent_agg.rename(columns={'trade_date': 'date'}, inplace=True)

print(f"Sentiment aggregated: {len(sent_agg):,} ticker-date pairs")

# Merge with prices
prices = prices.merge(sent_agg, on=['ticker', 'date'], how='left')

# Fill missing: no news = neutral
for col in ['sent_mean', 'sent_max', 'sent_min', 'sent_confidence']:
    prices[col] = prices[col].fillna(0)
prices['sent_count'] = prices['sent_count'].fillna(0).astype(int)

# Rolling 7-day sentiment (per ticker, without groupby.apply)
prices = prices.sort_values(['ticker', 'date'])
sent_7d = []
for ticker in prices['ticker'].unique():
    mask = prices['ticker'] == ticker
    sent_7d.append(prices.loc[mask, 'sent_mean'].rolling(7, min_periods=1).mean())
prices['sent_mean_7d'] = pd.concat(sent_7d)

# Announcement count z-score (ticker-level)
ticker_stats = prices.groupby('ticker')['sent_count'].agg(['mean', 'std']).reset_index()
ticker_stats.columns = ['ticker', 'count_mean', 'count_std']
prices = prices.merge(ticker_stats, on='ticker', how='left')
prices['sent_count_zscore'] = (prices['sent_count'] - prices['count_mean']) / prices['count_std'].replace(0, 1)
prices.drop(columns=['count_mean', 'count_std'], inplace=True)

print(f"Sentiment features added.")
print(f"Days with any sentiment: {(prices['sent_count'] > 0).sum():,} / {len(prices):,}")

Sentiment aggregated: 820 ticker-date pairs
Sentiment features added.
Days with any sentiment: 796 / 37,506


In [36]:
# Anomaly label: uç hareket VE sektörden ayrışma
# - z-score > 1.96: hisse kendi geçmişine göre uç
# - |return_vs_sector| > 0.03: hareket sektör geneline atfedilemez
#   (makro şokları eler, hisseye özgü hareketi yakalar)
prices = prices.sort_values(['ticker', 'date'])

return_zscore = []
is_anomaly = []

for ticker in prices['ticker'].unique():
    mask = prices['ticker'] == ticker
    returns = prices.loc[mask, 'return_1d']
    rolling_mean = returns.rolling(30, min_periods=20).mean()
    rolling_std = returns.rolling(30, min_periods=20).std()
    zscore = (returns - rolling_mean) / rolling_std.replace(0, np.nan)
    return_zscore.append(zscore)
    
    # Yeni: z-score uç VE sektörden ayrışma
    anomaly = (
        (zscore.abs() > 1.96) & 
        (prices.loc[mask, 'return_vs_sector'].abs() > 0.03)
    ).astype(int)
    is_anomaly.append(anomaly)

prices['return_zscore'] = pd.concat(return_zscore)
prices['is_anomaly'] = pd.concat(is_anomaly)

anomaly_rate = prices['is_anomaly'].mean()
print(f"Anomaly rate: {anomaly_rate:.1%} ({prices['is_anomaly'].sum():,} / {len(prices):,})")
print(f"\nPer-ticker anomaly count (top 10):")
print(prices.groupby('ticker')['is_anomaly'].sum().sort_values(ascending=False).head(10))

Anomaly rate: 1.3% (495 / 37,506)

Per-ticker anomaly count (top 10):
ticker
GUBRF    53
SASA     49
TKFEN    45
HALKB    42
KONTR    39
KRDMD    28
TUPRS    25
PETKM    24
TOASO    20
VAKBN    19
Name: is_anomaly, dtype: int64


In [37]:
FEATURE_COLS = [
    # Price features
    'return_5d', 'return_10d', 'return_20d',
    'volatility_10d', 'volatility_20d',
    'volume_ratio_20d',
    # Sentiment
    'sent_mean', 'sent_max', 'sent_min', 'sent_count', 'sent_confidence',
    'sent_mean_7d', 'sent_count_zscore',
]

LABEL = 'is_anomaly'
META_COLS = ['ticker', 'date', 'sector', 'Close']

# Drop rows with NaN in features (first ~20 rows per ticker due to rolling windows)
feat = prices[META_COLS + FEATURE_COLS + [LABEL, 'return_zscore']].dropna(subset=FEATURE_COLS)

print(f"Final feature matrix: {feat.shape[0]:,} rows × {len(FEATURE_COLS)} features")
print(f"Date range: {feat['date'].min().date()} → {feat['date'].max().date()}")
print(f"Anomaly rate: {feat[LABEL].mean():.1%}")
print(f"Tickers: {feat['ticker'].nunique()}")

# Save
feat.to_parquet("/Users/ulasmerttoy/Desktop/bist30/bist-30-assistant/data/processed/features.parquet", index=False)
print(f"\n✅ Saved to data/processed/features.parquet")

Final feature matrix: 36,906 rows × 13 features
Date range: 2021-06-01 → 2026-04-29
Anomaly rate: 1.3%
Tickers: 30

✅ Saved to data/processed/features.parquet


In [38]:
print("Feature summary statistics:")
print(feat[FEATURE_COLS].describe().round(3).to_string())

print(f"\n\nCorrelation with anomaly label (top features):")
corr = feat[FEATURE_COLS + [LABEL]].corr()[LABEL].drop(LABEL).abs().sort_values(ascending=False)
for feat_name, c in corr.head(10).items():
    print(f"  {feat_name:25s} {c:.3f}")

print(f"\n\nSentiment on anomaly days vs normal days:")
for col in ['sent_mean', 'sent_count', 'sent_mean_7d']:
    anom_val = feat[feat[LABEL]==1][col].mean()
    norm_val = feat[feat[LABEL]==0][col].mean()
    print(f"  {col:20s}  anomaly={anom_val:.4f}  normal={norm_val:.4f}")

Feature summary statistics:
       return_5d  return_10d  return_20d  volatility_10d  volatility_20d  volume_ratio_20d  sent_mean   sent_max   sent_min  sent_count  sent_confidence  sent_mean_7d  sent_count_zscore
count  36906.000   36906.000   36906.000       36906.000       36906.000         36906.000  36906.000  36906.000  36906.000   36906.000        36906.000     36906.000          36906.000
mean       0.012       0.024       0.048           0.026           0.027             1.014      0.005      0.005      0.004       0.028            0.015         0.005              0.002
std        0.069       0.101       0.147           0.012           0.011             0.474      0.056      0.060      0.057       0.217            0.101         0.025              1.008
min       -0.544      -0.606      -0.669           0.000           0.005             0.000     -0.850     -0.850     -0.850       0.000            0.000        -0.143             -0.258
25%       -0.028      -0.037      -0.046  